In [22]:
from collections import Counter
from datetime import datetime, timedelta

from pathlib import Path


import numpy as np
import pandas as pd

import plotly.graph_objects as go

from plotly.colors import qualitative, sample_colorscale

from plotly.subplots import make_subplots

from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor

from sklearn.metrics import mean_pinball_loss, mean_squared_error

from sklearn.model_selection import train_test_split


from src.utils import (
    formatTimedelta,
    getFilesByDate,
    isOdd,
    parallelizeFunction,
    sortStrNumbers,
)

from src.visualizacion.visualizaciones import setHoverInfo

In [2]:
dir_data = Path(r"data/shiny/xsiv/PRO/")
grandes = list(dir_data.iterdir())
grandes_names = [el.name for el in grandes]

In [ ]:
def cargarDatos(
    station_name: str,
    start: str = "2024-05-24",
    end: str = "2024-05-29",
):
    start = pd.to_datetime(start)
    end = pd.to_datetime(end)
    use_station = np.where(np.array(grandes_names) == station_name)[0][0]
    files = list(zip(*getFilesByDate(grandes[use_station], start=start, end=end)))[0]
    # files = list(grandes[use_station].rglob("*.*"))
    if not files or files is None:
        return
    # files, _ = list(zip(*files))
    files = [el for el in files if el.suffix == ".xlsx"]
    dfs = [
        el
        for el in parallelizeFunction(pd.read_excel, files, show_progress=False)
        if el is not None and not el.empty
    ]
    if not dfs:
        return
    df = pd.concat(dfs).drop_duplicates()
    df["Ocupación (segundos)"] = (
        (df["FinOcupación"] - df["InicioOcupación"]).dt.total_seconds().fillna(0)
    )
    df["OcupaciónPlanificada (segundos)"] = (
        (df["SalidaPlanificada"] - df["LlegadaPlanificada"])
        .dt.total_seconds()
        .fillna(0)
    )
    fechas_movs = df[
        ["ALTA", "APROXIMACIÓN", "MANIOBRALLEGADA", "LLEGADA", "SALIDA"]
        + ["MANIOBRASALIDA", "FIN", "BAJA", "MANIOBRA"]
    ].apply(pd.to_datetime)
    df["Inicio"] = fechas_movs.min(axis=1)
    df["Fin"] = fechas_movs.max(axis=1)
    return df.reset_index(drop=True)

In [ ]:
df = cargarDatos("zaragoza_delicias", start="2024-05-01", end="2024-06-29")

In [ ]:
df.columns

In [ ]:
df["Vía planificada"].unique()

In [14]:
df_test = df.dropna(
    subset=["Inicio Ocupación", "Llegada planificada"], how="any"
).copy()

df_test["Retraso"] = (
    df_test["Inicio Ocupación"] - df_test["Llegada planificada"]
).apply(lambda x: x.total_seconds())

df_test["Fecha planificada (norm)"] = df_test["Llegada planificada"].apply(
    lambda x: x + (datetime.today().date() - x.date())
)
df_test["Fecha planificada (norm)"] = np.where(
    df_test["Fecha planificada (norm)"]
    < pd.to_datetime(datetime.today().date()) + timedelta(hours=3),
    df_test["Fecha planificada (norm)"] + timedelta(days=1),
    df_test["Fecha planificada (norm)"],
)
df_test["D.Semana"] = df_test["Llegada planificada"].dt.day_of_week.astype(str)
df_test["Impar"] = df_test["T1"].apply(isOdd)

### Matriz confusión

In [ ]:
df_test[["Vía", "Vía planificada"]].groupby(["Vía", "Vía planificada"]).size().unstack().columns

In [ ]:
df_test[["Vía", "Vía planificada"]].groupby(["Vía", "Vía planificada"]).size().unstack().index

In [165]:
# df_test["Total"] = 1
# conf = df_test[["Vía", "Vía planificada", "Total"]].pivot_table(
#     values="Total",
#     index="Vía",
#     columns="Vía planificada",
#     aggfunc="sum",
# )
conf = df_test[["Vía", "Vía planificada"]].groupby(["Vía", "Vía planificada"]).size().unstack()
vias = sortStrNumbers(set(list(conf.index) + list(conf.columns)))
conf[[v for v in vias if v not in conf.columns]] = None
for v in vias:
    if v not in conf.index:
        conf.loc[v] = None
conf = conf.loc[sortStrNumbers(conf.index), sortStrNumbers(conf.columns)]

In [167]:
hover_cols = ["Vía", "Vía planificada", "Total"]
flen = max([len(c) for c in hover_cols]) + 2
fd_len = min(
    conf.fillna("").map(lambda x: len(f"{x}"), na_action="ignore").max().max(),
    8,
)
# hover_info = [
#     [f"Vía: {v}<br>Vía planificada: {vp}<br>Total: {t:.0f}" for vp, t in el.items()]
#     for v, el in conf.to_dict(orient="index").items()
# ]
hover_info = [
    [
        f"{'Vía':<{flen}}{v:>{fd_len}}<br>{'Vía planificada':<{flen}}{vp:>{fd_len}}<br>{'Total':<{flen}}{t:>{fd_len}.0f}"
        for vp, t in el.items()
    ]
    for v, el in conf.fillna(0).to_dict(orient="index").items()
]
# hover_info

In [ ]:
hover_info

In [ ]:
conf.fillna(0).values.sum()/max(conf.shape)

In [ ]:
fig = go.Figure(
    data=go.Heatmap(
        z=conf.values,
        x=conf.columns,
        y=conf.index,
        hoverongaps=False,
        hoverinfo="text",
        hovertext=hover_info,
        # text=conf.values,
        # texttemplate="%{text}",
        # textfont={"size": 20},
        # zmid=conf.fillna(0).values.mean(),
        zmin=0,
        zmax=conf.fillna(0).values.sum() / max(conf.shape),
        # colorscale=[
        #     [0.0, "rgb(165,0,38)"],
        #     [0.1111111111111111, "rgb(215,48,39)"],
        #     [0.2222222222222222, "rgb(244,109,67)"],
        #     [0.3333333333333333, "rgb(253,174,97)"],
        #     [0.4444444444444444, "rgb(254,224,144)"],
        #     [0.5555555555555556, "rgb(224,243,248)"],
        #     [0.6666666666666666, "rgb(171,217,233)"],
        #     [0.7777777777777778, "rgb(116,173,209)"],
        #     [0.8888888888888888, "rgb(69,117,180)"],
        #     [1.0, "rgb(49,54,149)"],
        # ],
    )
)
# add title
fig.update_layout(
    title_text=f"<b>Matriz confusión</b>",
    # xaxis = dict(title='x'),
    # yaxis = dict(title='x')
)

# add custom xaxis title
fig.add_annotation(
    dict(
        font=dict(color="black", size=14),
        x=0.5,
        y=-0.15,
        showarrow=False,
        text="Vía planificada",
        xref="paper",
        yref="paper",
    )
)

# add custom yaxis title
fig.add_annotation(
    dict(
        font=dict(color="black", size=14),
        x=-0.15,
        y=0.5,
        showarrow=False,
        text="Vía real",
        textangle=-90,
        xref="paper",
        yref="paper",
    )
)

# adjust margins to make room for yaxis title
fig.update_layout(
    margin=dict(t=50, l=100),
    xaxis_tickangle=0,
    yaxis_tickangle=0,
    # width=1000,
    # height=800,
    hoverlabel=dict(bgcolor="white", font_size=16, font_family="consolas"),
)

# add colorbar
fig["data"][0]["showscale"] = True
fig.show()

In [156]:
fig.write_html("a.html")

### Retrasos

In [6]:
xrange_time = pd.date_range(
    datetime.today().date(),
    datetime.today().date() + timedelta(days=1),
    freq="s",
    inclusive="left",
)
xrange_time = np.sort(
    np.where(
        xrange_time < pd.to_datetime(datetime.today().date()) + timedelta(hours=3),
        xrange_time + timedelta(days=1),
        xrange_time,
    )
)

In [8]:
def getXY(df: pd.DataFrame):
    # X = np.atleast_2d(df_test[["Fecha planificada (norm)", "Vía"]].values)
    X = np.hstack(
        [
            df[["Fecha planificada (norm)"]].values,
            df[["Impar", "Vía", "ProductoT1", "D.Semana"]].values,
        ]
    )
    y = df["Retraso"].values

    X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)
    return X_train, X_test, y_train, y_test


def getEstimators(
    X_train, y_train, common_params, conf=0.05
) -> dict[str, HistGradientBoostingRegressor]:
    all_models = {}
    # common_params = dict(
    #     learning_rate=0.05,
    #     # n_estimators=200,
    #     max_iter=200,
    #     # min_samples_leaf=5,
    #     # min_samples_split=9,
    #     max_leaf_nodes=64,
    # )
    categorical_features = [
        False,
        False,
        True,
        True,
        True,
    ]
    # Intervalos
    for alpha in [conf, 0.5, 1 - conf]:
        # gbr = GradientBoostingRegressor(loss="quantile", alpha=alpha, **common_params)
        gbr = HistGradientBoostingRegressor(
            loss="quantile",
            # categorical_features="from_dtype",
            categorical_features=categorical_features,
            quantile=alpha,
            **common_params,
        )
        all_models[f"q {alpha:.2f}"] = gbr.fit(X_train, y_train)

    # MSE
    gbr_ls = HistGradientBoostingRegressor(
        loss="squared_error",
        # categorical_features="from_dtype",
        categorical_features=categorical_features,
        **common_params,
    )
    all_models["mse"] = gbr_ls.fit(X_train, y_train)

    return all_models

In [70]:
def addIntervalTrace(
    x,
    y_upper,
    y_lower,
    name,
    hover_info,
    color="rgba(0,0,0,0)",
    showlegend: bool = True,
    # opacity: float = 1,
    # width: float = 0,
    # dash=None,
):
    data = []
    data.append(
        go.Scatter(
            x=x,
            y=y_upper,
            mode="lines",
            name=name,
            showlegend=False,
            legendgroup=name,
            line=dict(color=color),
            # fill="tonexty",
            hoverinfo="text",
            hovertext=hover_info,
        )
    )
    data.append(
        go.Scatter(
            x=x,
            y=y_lower,
            mode="lines",
            name=name,
            showlegend=showlegend,
            legendgroup=name,
            line=dict(color=color),
            fill="tonexty",
            # fillcolor="",
            hoverinfo="text",
            hovertext=hover_info,
        )
    )
    return data

In [ ]:
df_test.shape

In [10]:
marg = 0.05  # 1-(2*conf)

X_train, X_test, y_train, y_test = getXY(df_test)
common_params = dict(
    learning_rate=0.05,
    # n_estimators=200,
    max_iter=100,
    min_samples_leaf=5,
    # min_samples_split=9,
    # max_leaf_nodes=64,
    max_leaf_nodes=31,
)

# all_models = getEstimators(X_train, y_train, conf)
all_models = getEstimators(
    np.concatenate([X_train, X_test]),
    np.concatenate([y_train, y_test]),
    common_params,
    marg,
)

In [55]:
def getTracesVia(
    via: str,
    real_x: np.ndarray[np.datetime64] = None,
    real_y: np.ndarray[float] = None,
    xx: np.ndarray[np.datetime64] = None,
    y_upper: np.ndarray[float] = None,
    y_lower: np.ndarray[float] = None,
    conf: float = None,
    y_pred: np.ndarray[float] = None,
    y_med: np.ndarray[float] = None,
):
    data = []
    if xx is not None:
        if y_upper is not None and y_lower is not None and conf is not None:
            data.append(
                go.Scatter(
                    x=xx.ravel(),
                    y=y_upper,
                    mode="lines",
                    name=f"Predicción intervalo {(1-2*conf)*100}% vía {via}",
                    showlegend=False,
                    legendgroup=f"Predicción intervalo {(1-2*conf)*100}% vía {via}",
                    line=dict(color="rgba(0,0,0,0)"),
                    # fill="tonexty",
                )
            )
            data.append(
                go.Scatter(
                    x=xx.ravel(),
                    y=y_lower,
                    mode="lines",
                    name=f"Predicción intervalo {(1-2*conf)*100}% vía {via}",
                    showlegend=True,
                    legendgroup=f"Predicción intervalo {(1-2*conf)*100}% vía {via}",
                    line=dict(color="rgba(0,0,0,0)"),
                    fill="tonexty",
                    # fillcolor="",
                )
            )
        if y_pred is not None:
            data.append(
                go.Scatter(
                    x=xx.ravel(),
                    y=y_pred,
                    mode="lines",
                    name=f"Media vía {via}",
                    showlegend=True,
                    legendgroup=f"Pred",
                )
            )
        if y_med is not None:
            data.append(
                go.Scatter(
                    x=xx.ravel(),
                    y=y_med,
                    mode="lines",
                    name=f"Mediana vía {via}",
                    showlegend=True,
                    legendgroup=f"Pred",
                )
            )
    if real_x is not None and real_y is not None:
        data.append(
            go.Scatter(
                # line=dict(color="black", width=row["Value"]),
                # label=nodes["label"],
                x=real_x,
                y=real_y,
                mode="markers",
                hoverinfo="text",
                name=f"Vía {via}",
                showlegend=True,
                legendgroup=f"{via}",
            )
        )
    return data

In [74]:
def predict(
    all_models: dict[str, HistGradientBoostingRegressor],
    config: dict,
    marg: float,
    mse: bool = True,
    med: bool = True,
    prd: bool = True,
):
    x_pred = np.array(
        [list(xrange_time)] + [[el] * len(xrange_time) for el in config.values()]
    ).T

    predictions = dict()
    if mse:
        predictions["y_pred"] = all_models["mse"].predict(x_pred)
    if med:
        predictions["y_med"] = all_models["q 0.50"].predict(x_pred)
    if prd:
        predictions["y_lower"] = all_models[f"q {marg:.2f}"].predict(x_pred)
        predictions["y_upper"] = all_models[f"q {1-marg:.2f}"].predict(x_pred)
    return predictions


# Predicción
# ["Fecha planificada (norm)", "Impar", "Vía", "ProductoT1", "D.Semana"]
config = dict(
    odd=True,
    via="1",
    prod="CERCANIAS",  # "CERCANIAS", "AVE", "ALVIA", "Material Vacio", "AVANT", "OUIGO", "MD"
    dsem="2",  # "0", "1", "2", "3", "4", "5", "6" (lunes-domingo)
)
config = dict(
    odd=True,
    via="1",
    prod="ALVIA",  # "CERCANIAS", "AVE", "ALVIA", "Material Vacio", "AVANT", "OUIGO", "MD"
    dsem="2",  # "0", "1", "2", "3", "4", "5", "6" (lunes-domingo)
)
mse = True
med = True
prd = True
predictions = predict(
    all_models,
    config,
    marg,
    mse=mse,
    med=med,
    prd=prd,
)
xx = np.array(list(xrange_time))
y_pred = predictions.get("y_pred")
y_med = predictions.get("y_med")
y_lower = predictions.get("y_lower")
y_upper = predictions.get("y_upper")
# Real
df_rep = df_test[df_test["Vía"] == config["via"]]
real_x = df_rep["Fecha planificada (norm)"]
real_y = df_rep["Retraso"]

In [75]:
aux_df = pd.DataFrame(
    {"Hora": xx, "Inferior": y_lower, "Superior": y_upper, "Vía": config["via"]}
)
aux_df[["Inferior", "Superior"]] = (
    aux_df[["Inferior", "Superior"]].map(lambda x: f"{formatTimedelta(x)}").values
)
aux_df["hover_info"] = setHoverInfo(aux_df, ["Hora", "Inferior", "Superior", "Vía"])

In [76]:
traces = []
if prd:
    name = f"Predicción intervalo {(1-2*marg)*100}% vía {config['via']}"
    aux_df = pd.DataFrame(
        {
            "Hora": xx,
            "Inferior (segundos)": y_lower,
            "Superior (segundos)": y_upper,
            "Vía": config["via"],
        }
    )
    aux_df[["Inferior", "Superior"]] = (
        aux_df[["Inferior (segundos)", "Superior (segundos)"]]
        .map(lambda x: f"{formatTimedelta(x)}")
        .values
    )
    aux_df["hover_info"] = setHoverInfo(aux_df, ["Hora", "Inferior", "Superior", "Vía"])
    traces.extend(
        addIntervalTrace(
            aux_df["Hora"],
            aux_df["Superior (segundos)"],
            aux_df["Inferior (segundos)"],
            name,
            aux_df["hover_info"],
        )
    )

if med:
    aux_df = pd.DataFrame(
        {"Hora": xx, "Mediana (segundos)": y_med, "Vía": config["via"]}
    )
    aux_df["Mediana"] = (
        aux_df["Mediana (segundos)"].apply(lambda x: f"{formatTimedelta(x)}").values
    )
    aux_df["hover_info"] = setHoverInfo(aux_df, ["Hora", "Mediana", "Vía"])
    traces.append(
        go.Scatter(
            x=aux_df["Hora"],
            y=aux_df["Mediana (segundos)"],
            mode="lines",
            name=f"Mediana vía {config['via']}",
            showlegend=True,
            legendgroup=f"Pred",
            # line=dict(color=color),
            hoverinfo="text",
            hovertext=aux_df["hover_info"],
        )
    )
if mse:
    aux_df = pd.DataFrame(
        {"Hora": xx, "Media (segundos)": y_pred, "Vía": config["via"]}
    )
    aux_df["Media"] = (
        aux_df["Media (segundos)"].apply(lambda x: f"{formatTimedelta(x)}").values
    )
    aux_df["hover_info"] = setHoverInfo(aux_df, ["Hora", "Media", "Vía"])
    traces.append(
        go.Scatter(
            x=aux_df["Hora"],
            y=aux_df["Media (segundos)"],
            mode="lines",
            name=f"Media vía {config['via']}",
            showlegend=True,
            legendgroup=f"Pred",
            # line=dict(color=color),
            hoverinfo="text",
            hovertext=aux_df["hover_info"],
        )
    )


aux_df = pd.DataFrame(
    {"Hora": real_x, "Retraso (segundos)": real_y, "Vía": config["via"]}
)
aux_df["Retraso"] = (
    aux_df["Retraso (segundos)"].apply(lambda x: f"{formatTimedelta(x)}").values
)
aux_df["hover_info"] = setHoverInfo(aux_df, ["Hora", "Retraso", "Vía"])
traces.append(
    go.Scatter(
        # line=dict(color="black", width=row["Value"]),
        # label=nodes["label"],
        x=aux_df["Hora"],
        y=aux_df["Retraso (segundos)"],
        mode="markers",
        name=f"Vía {config['via']}",
        showlegend=True,
        legendgroup=f"{config['via']}",
        hoverinfo="text",
        hovertext=aux_df["hover_info"],
    )
)

In [77]:
layout = go.Layout(
    hoverlabel=dict(bgcolor="white", font_size=16, font_family="consolas"),
    xaxis=dict(
        autorange=True,
        rangeslider=dict(
            visible=True,
            thickness=0.1,
        ),
        type="date",
        tickformat="%H:%M:%S",
        fixedrange=False,
    ),
    yaxis=dict(
        type="linear",
        # type="log",
        autorange=True,
        fixedrange=False,
    ),
    legend=dict(
        groupclick="toggleitem",
        # groupclick="togglegroup",
        yanchor="middle",
        y=0.5,
    ),
)


fig = go.Figure(data=traces, layout=layout)


# # fig.write_html("aver2.html")


# fig.show()

In [ ]:
fig.show()

In [ ]:
fig.show()

In [50]:
n = 2
X_train, X_test, y_train, y_test = getXY(df_test)
common_params = dict(
    learning_rate=0.05,
    # n_estimators=200,
    max_iter=100,
    min_samples_leaf=5,
    # min_samples_split=9,
    # max_leaf_nodes=64,
    max_leaf_nodes=31,
)

conf = 0.05  # 1-(2*conf)
# all_models = getEstimators(X_train, y_train, conf)
all_models = getEstimators(
    np.concatenate([X_train, X_test]),
    np.concatenate([y_train, y_test]),
    common_params,
    conf,
)

fig = make_subplots(rows=n, shared_xaxes=True)
yaxes = {}
for i, via in enumerate(df_test["Vía"].value_counts()[:n].index):
    x_pred = np.array(
        [
            list(xrange_time),
            [via] * len(xrange_time),
            ["C"] * len(xrange_time),
        ]
    ).T
    xx = np.array(list(xrange_time))
    y_pred = all_models["mse"].predict(x_pred)
    y_lower = all_models[f"q {conf:.2f}"].predict(x_pred)
    y_upper = all_models[f"q {1-conf:.2f}"].predict(x_pred)
    y_med = all_models["q 0.50"].predict(x_pred)

    df_rep = df_test[df_test["Vía"] == via]
    real_x = df_rep["Fecha planificada (norm)"]
    real_y = df_rep["Retraso"]
    traces = getTracesVia(
        via=via,
        real_x=real_x,
        real_y=real_y,
        xx=xx,
        y_upper=y_upper,
        y_lower=y_lower,
        conf=conf,
        y_pred=y_pred,
        y_med=y_med,
    )
    # fig.add_traces(traces, rows=[i + 1] * 5, cols=[1] * 5)
    fig.add_traces(traces, rows=i + 1, cols=1)
    if not i:
        yaxes["yaxis"] = dict(
            type="linear",
            # type="log",
            autorange=True,
            fixedrange=False,
        )
    else:
        yaxes[f"yaxis{i+1}"] = dict(
            type="linear",
            # type="log",
            autorange=True,
            fixedrange=False,
        )

In [ ]:
layout = go.Layout(
    xaxis2=dict(
        autorange=True,
        rangeslider=dict(
            visible=True,
            thickness=0.1,
        ),
        type="date",
        tickformat="%H:%M:%S",
        fixedrange=False,
    ),
    **yaxes,
    # yaxis=dict(
    #     type="linear",
    #     # type="log",
    #     autorange=True,
    #     fixedrange=False,
    # ),
    legend=dict(
        groupclick="toggleitem",
        # groupclick="togglegroup",
        yanchor="middle",
        y=0.5,
    ),
)


fig.update_layout(layout)


fig.write_html("cha.html")
fig.show()

In [ ]:
fig.layout

In [17]:
# n = 1
# fig = make_subplots(rows=n, shared_xaxes=True)
# for i, via in enumerate(df_test["Vía"].value_counts()[:n].index):
#     x_pred = np.array([list(xrange_time), [via] * len(xrange_time)]).T
#     y_pred = all_models["mse"].predict(x_pred)
#     y_lower = all_models["q 0.05"].predict(x_pred)
#     y_upper = all_models["q 0.95"].predict(x_pred)
#     y_med = all_models["q 0.50"].predict(x_pred)

#     data = []
#     data.append(
#         go.Scatter(
#             x=xx.ravel(),
#             y=y_upper,
#             mode="lines",
#             name=f"Predicción intervalo 90% vía {via}",
#             showlegend=True,
#             legendgroup=f"Predicted 90% interval",
#             line=dict(color="rgba(0,0,0,0)"),
#         )
#     )
#     data.append(
#         go.Scatter(
#             x=xx.ravel(),
#             y=y_lower,
#             mode="lines",
#             name=f"Predicted 90% interval",
#             showlegend=False,
#             legendgroup=f"Predicción intervalo 90% vía {via}",
#             line=dict(color="rgba(0,0,0,0)"),
#             fill="tonexty",
#             # fillcolor="",
#         )
#     )
#     df_rep = df_test[df_test["Vía"] == via]
#     data.append(
#         go.Scatter(
#             # line=dict(color="black", width=row["Value"]),
#             # label=nodes["label"],
#             x=df_rep["Fecha planificada (norm)"],
#             y=df_rep["Retraso"],
#             mode="markers",
#             hoverinfo="text",
#             textfont=dict(family="calibri", size=18),
#             name=f"Vía {via}",
#             showlegend=True,
#             legendgroup=f"{via}",
#         )
#     )

#     data.append(
#         go.Scatter(
#             x=xx.ravel(),
#             y=y_pred,
#             mode="lines",
#             name=f"Media vía {via}",
#             showlegend=True,
#             legendgroup=f"Pred",
#         )
#     )
#     data.append(
#         go.Scatter(
#             x=xx.ravel(),
#             y=y_med,
#             mode="lines",
#             name=f"Mediana vía {via}",
#             showlegend=True,
#             legendgroup=f"Pred",
#         )
#     )
#     fig.add_traces(data, rows=[i + 1] * 5, cols=[1] * 5)

In [ ]:
# layout = go.Layout(
#     xaxis=dict(
#         autorange=True,
#         rangeslider=dict(
#             visible=True,
#             thickness=0.1,
#         ),
#         type="date",
#         tickformat="%H:%M:%S",
#         fixedrange=False,
#     ),
#     # yaxis=dict(type="log"),
#     yaxis=dict(type="linear"),
#     legend=dict(
#         groupclick="toggleitem",
#         # groupclick="togglegroup",
#         yanchor="middle",
#         y=0.5,
#     ),
# )
# fig = fig.update_layout(layout)
# fig.write_html("aver.html")


# # fig.show()